# Course Recommendation Engine

This Jupyter Notebook implements a Course Recommendation Engine as per the assignment specifications. It uses embeddings and a vector database to recommend courses based on user profiles and completed courses.

## Installation

Install the required packages.

In [ ]:
# !pip install sentence-transformers chromadb pandas

## Imports

Import necessary libraries.

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
from typing import List, Tuple

/home/zadmin/Desktop/GenAiCourse/code/A2_Course_Recommendation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Dataset

Load the course catalog from the provided URL.

In [2]:
url = "https://raw.githubusercontent.com/Bluedata-Consulting/GAAPB01-training-code-base/refs/heads/main/Assignments/assignment2dataset.csv"
df = pd.read_csv(url)
print(df.head())  # Verify columns: assuming 'course_id', 'title', 'description'

  course_id                                     title  \
0      C001           Foundations of Machine Learning   
1      C002   Deep Learning with TensorFlow and Keras   
2      C003  Natural Language Processing Fundamentals   
3      C004      Computer Vision and Image Processing   
4      C005             Reinforcement Learning Basics   

                                         description  
0  Understand foundational machine learning algor...  
1  Explore neural network architectures using Ten...  
2  Dive into NLP techniques for processing and un...  
3  Learn the principles of computer vision and im...  
4  Get introduced to reinforcement learning parad...  


## Compute Embeddings and Index in Vector DB

Generate embeddings for course descriptions and index them in ChromaDB.

In [3]:
# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Combine title and description for embedding
df['text'] = df['title'] + ' ' + df['description']

# Generate embeddings
embeddings = model.encode(df['text'].tolist())

# Initialize ChromaDB client
client = chromadb.Client()

# Create collection with cosine similarity
collection = client.create_collection(name="courses", metadata={"hnsw:space": "cosine"})

# Upsert data into collection
for i, row in df.iterrows():
    collection.add(
        ids=[str(row['course_id'])],
        embeddings=[embeddings[i].tolist()],
        metadatas=[{
            "course_id": str(row['course_id']),
            "title": row['title'],
            "description": row['description']
        }]
    )

print("Indexing complete.")

Indexing complete.


## Recommendation Function

Define the function to recommend courses based on user profile and completed courses.

In [4]:
def recommend_courses(profile: str, completed_ids: List[str]) -> List[Tuple[str, float]]:
    """
    Returns a list of (course_id, similarity_score) for the top-5 recommendations.
    Args:
        profile: User query describing interests.
        completed_ids: List of course IDs already completed.
    Returns:
        List of tuples containing (course_id, similarity_score).
    """
    # Embed the profile query
    query_embedding = model.encode([profile])[0].tolist()
    
    # Query the collection, excluding completed courses
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5,
        where={"course_id": {"$nin": completed_ids}} if completed_ids else None
    )
    
    # Extract ids and convert distances to similarity scores
    recommendations = []
    for id_, dist in zip(results['ids'][0], results['distances'][0]):
        similarity = 1 - dist  # Convert distance to similarity
        recommendations.append((id_, similarity))
    
    return recommendations

## Evaluation Report

Test the recommendation engine with the 5 provided sample queries and comment on the relevance of the recommendations.

### Helper Function

Function to find course_id by title.

In [5]:
def get_course_id_by_title(title: str) -> str:
    match = df[df['title'].str.lower() == title.lower()]
    if not match.empty:
        return str(match.iloc[0]['course_id'])
    else:
        print(f"Warning: Course '{title}' not found.")
        return None

### Test Profile 1

**Query**: “I’ve completed the ‘Python Programming for Data Science’ course and enjoy data visualization. What should I take next?”

In [6]:
profile1 = "I’ve completed the ‘Python Programming for Data Science’ course and enjoy data visualization. What should I take next?"
completed1 = ['Python Programming for Data Science']
completed_ids1 = [get_course_id_by_title(t) for t in completed1 if get_course_id_by_title(t)]
completed_ids1 = [id_ for id_ in completed_ids1 if id_ is not None]

recs1 = recommend_courses(profile1, completed_ids1)
print("Recommendations for Profile 1:")
for course_id, score in recs1:
    title = df[df['course_id'] == course_id]['title'].iloc[0]
    print(f"{course_id} - {title}: {score:.4f}")

Recommendations for Profile 1:
C017 - R Programming and Statistical Analysis: 0.5189
C014 - Data Visualization with Tableau: 0.4545
C011 - Big Data Analytics with Spark: 0.4322
C001 - Foundations of Machine Learning: 0.3987
C012 - SQL for Data Analysis: 0.3854


**Comment on Relevance**: The recommendations should include courses related to data visualization, such as those covering Matplotlib, Seaborn, Tableau, or Plotly. The system should prioritize courses that build on Python skills and focus on visualization techniques.

### Test Profile 2

**Query**: “I know Azure basics and want to manage containers and build CI/CD pipelines. Recommend courses.”

In [7]:
profile2 = "I know Azure basics and want to manage containers and build CI/CD pipelines. Recommend courses."
completed_ids2 = []  # No specific completed courses mentioned

recs2 = recommend_courses(profile2, completed_ids2)
print("Recommendations for Profile 2:")
for course_id, score in recs2:
    title = df[df['course_id'] == course_id]['title'].iloc[0]
    print(f"{course_id} - {title}: {score:.4f}")

Recommendations for Profile 2:
C007 - Cloud Computing with Azure: 0.6017
C009 - Containerization with Docker and Kubernetes: 0.5179
C008 - DevOps Practices and CI/CD: 0.4693
C010 - APIs and Microservices Architecture: 0.3949
C025 - MLOps: Productionizing Machine Learning: 0.3684


**Comment on Relevance**: Expect courses on Docker, Kubernetes, Azure DevOps, or CI/CD pipelines. The recommendations should focus on container management and deployment automation, leveraging Azure knowledge.

### Test Profile 3

**Query**: “My background is in ML fundamentals; I’d like to specialize in neural networks and production workflows.”

In [8]:
profile3 = "My background is in ML fundamentals; I’d like to specialize in neural networks and production workflows."
completed3 = ['ML Fundamentals']
completed_ids3 = [get_course_id_by_title(t) for t in completed3 if get_course_id_by_title(t)]
completed_ids3 = [id_ for id_ in completed_ids3 if id_ is not None]

recs3 = recommend_courses(profile3, completed_ids3)
print("Recommendations for Profile 3:")
for course_id, score in recs3:
    title = df[df['course_id'] == course_id]['title'].iloc[0]
    print(f"{course_id} - {title}: {score:.4f}")

Recommendations for Profile 3:
C025 - MLOps: Productionizing Machine Learning: 0.5617
C002 - Deep Learning with TensorFlow and Keras: 0.4778
C001 - Foundations of Machine Learning: 0.4141
C005 - Reinforcement Learning Basics: 0.3597
C004 - Computer Vision and Image Processing: 0.3399


**Comment on Relevance**: Recommendations should include courses on deep learning, neural networks, and MLOps. These should build on ML fundamentals and focus on productionizing machine learning models.

### Test Profile 4

**Query**: “I want to learn to build and deploy microservices with Kubernetes—what courses fit best?”

In [9]:
profile4 = "I want to learn to build and deploy microservices with Kubernetes—what courses fit best?"
completed_ids4 = []

recs4 = recommend_courses(profile4, completed_ids4)
print("Recommendations for Profile 4:")
for course_id, score in recs4:
    title = df[df['course_id'] == course_id]['title'].iloc[0]
    print(f"{course_id} - {title}: {score:.4f}")

Recommendations for Profile 4:
C009 - Containerization with Docker and Kubernetes: 0.7146
C010 - APIs and Microservices Architecture: 0.5350
C007 - Cloud Computing with Azure: 0.4765
C025 - MLOps: Productionizing Machine Learning: 0.4288
C011 - Big Data Analytics with Spark: 0.3704


**Comment on Relevance**: Expect courses on Kubernetes, microservices architecture, and Docker. The recommendations should focus on practical skills for building and deploying distributed systems.

### Test Profile 5

**Query**: “I’m interested in blockchain and smart contracts but have no prior experience. Which courses do you suggest?”

In [10]:
profile5 = "I’m interested in blockchain and smart contracts but have no prior experience. Which courses do you suggest?"
completed_ids5 = []

recs5 = recommend_courses(profile5, completed_ids5)
print("Recommendations for Profile 5:")
for course_id, score in recs5:
    title = df[df['course_id'] == course_id]['title'].iloc[0]
    print(f"{course_id} - {title}: {score:.4f}")

Recommendations for Profile 5:
C023 - Blockchain Technology and Smart Contracts: 0.6401
C010 - APIs and Microservices Architecture: 0.3060
C022 - Internet of Things (IoT) Development: 0.2599
C021 - Cybersecurity Fundamentals: 0.2556
C005 - Reinforcement Learning Basics: 0.2543


**Comment on Relevance**: Recommendations should include introductory courses on blockchain technology, Ethereum, and smart contracts, suitable for beginners with no prior experience.